Data and plot for R(s) vs s

In [2]:
###for all together
import os
import numpy as np
import gsd.hoomd
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- Functions ----------------

def apply_pbc(positions, box):
    """Apply minimum-image PBC like previous scripts."""
    Lx, Ly, Lz, xy, xz, yz = box
    box_lengths = np.array([Lx, Ly, Lz])
    for i in range(positions.shape[0]):
        for dim in range(3):
            delta = positions[i, dim] - positions[0, dim]
            if delta > box_lengths[dim] / 2:
                positions[i, dim] -= box_lengths[dim]
            elif delta < -box_lengths[dim] / 2:
                positions[i, dim] += box_lengths[dim]
    return positions

def calculate_R_s(positions):
    N = len(positions)
    if N < 2:
        return np.array([])
    R_s = []
    for s in range(1, N):
        distances = np.linalg.norm(positions[s:] - positions[:-s], axis=1)
        R_s.append(np.mean(distances))
    return np.array(R_s)

def process_R_s(frames, Ca_index, n_chains, atoms_per_chain):
    R_s_all = []
    for frame in frames:
        positions = frame.particles.position
        types = frame.particles.typeid
        Ca_positions = positions[types == Ca_index]
        box = frame.configuration.box

        for i in range(n_chains):
            start = i * atoms_per_chain
            end = min((i + 1) * atoms_per_chain, len(Ca_positions))
            if start >= end:
                continue
            chain_positions = Ca_positions[start:end]
            if len(chain_positions) < 2:
                continue
            chain_positions = apply_pbc(chain_positions.copy(), box)
            R_s = calculate_R_s(chain_positions)
            if len(R_s) > 0:
                R_s_all.append(R_s)

    if len(R_s_all) == 0:
        return None, None, None, None, None, None

    R_s_all = np.array(R_s_all)
    R_s_mean = np.mean(R_s_all, axis=0)
    R_s_std = np.std(R_s_all, axis=0)
    s_values = np.arange(1, len(R_s_mean) + 1)
    log_s_values = np.log(s_values)
    log_R_s_mean = np.log(R_s_mean)
    log_R_s_std = R_s_std / R_s_mean
    return s_values, R_s_mean, R_s_std, log_s_values, log_R_s_mean, log_R_s_std

def save_plot(s_values, R_mean, R_std, log_s, log_R_mean, log_R_std, filename_csv, filename_png):
    if s_values is None:
        print(f"No data to save or plot for {filename_csv}")
        return
    df = pd.DataFrame({
        's': s_values,
        'R_g(s)_mean': R_mean,
        'R_g(s)_std': R_std,
        'log(s)': log_s,
        'log(R_g(s)_mean)': log_R_mean,
        'log(R_g(s)_std_rel)': log_R_std
    })
    df.to_csv(filename_csv, index=False)

    plt.figure(figsize=(8, 6))
    plt.errorbar(log_s, log_R_mean, yerr=log_R_std, fmt='o-', color='blue', ecolor='gray', capsize=3)
    plt.xlabel(r'$\log(s)$', fontsize=14)
    plt.ylabel(r'$\log(R(s))$', fontsize=14)
    plt.title(filename_png.replace('.png', ''))
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(filename_png, dpi=300)
    plt.close()

# ---------------- Main Script ----------------

# Bias values
bias_values = np.linspace(0.75, 1.0, 12)

# Three sets
sets = [
    {"name": "Full_length", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/"},
    {"name": "H0_H3", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/3de12d848c06a70b5668c64369550562/"},
    {"name": "H4_H6", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/7fd5f713b9bd4167bafd3d2d9378bc51/"}
]

n_chains = 64

for s in sets:
    print(f"\n=== Processing set: {s['name']} ===")
    output_dir = os.path.join(os.getcwd(), s['name'])
    os.makedirs(output_dir, exist_ok=True)

    for bias in bias_values:
        traj_file = os.path.join(s['base_path'], f'trajectory_bias_{bias}.gsd')
        if not os.path.isfile(traj_file):
            print(f"File {traj_file} not found. Skipping.")
            continue
        print(f"Processing trajectory for bias = {bias}")

        trajectory = gsd.hoomd.open(traj_file, 'r')
        all_types = trajectory[0].particles.types
        Ca_index = all_types.index("Ca")
        frames = trajectory[10000:20000]

        frame0 = frames[0]
        types = frame0.particles.typeid
        Ca_mask = (types == Ca_index)
        Ca_indices = np.where(Ca_mask)[0]
        total_Ca = len(Ca_indices)
        atoms_per_chain = total_Ca // n_chains

        s_vals, R_mean, R_std, log_s, log_R_mean, log_R_std = process_R_s(
            frames, Ca_index, n_chains, atoms_per_chain
        )

        bias_str = str(bias).replace('.', '_')
        csv_name = os.path.join(output_dir, f'R_vs_s_full_chain_bias_{bias_str}.csv')
        png_name = os.path.join(output_dir, f'R_vs_s_full_chain_bias_{bias_str}.png')

        save_plot(s_vals, R_mean, R_std, log_s, log_R_mean, log_R_std, csv_name, png_name)

print("All trajectories processed for all sets.")



=== Processing set: Full_length ===
Processing trajectory for bias = 0.75
Processing trajectory for bias = 0.7727272727272727
Processing trajectory for bias = 0.7954545454545454
Processing trajectory for bias = 0.8181818181818181
Processing trajectory for bias = 0.8409090909090909
Processing trajectory for bias = 0.8636363636363636
Processing trajectory for bias = 0.8863636363636364
Processing trajectory for bias = 0.9090909090909091
Processing trajectory for bias = 0.9318181818181819
Processing trajectory for bias = 0.9545454545454546
Processing trajectory for bias = 0.9772727272727273
Processing trajectory for bias = 1.0

=== Processing set: H0_H3 ===
Processing trajectory for bias = 0.75
Processing trajectory for bias = 0.7727272727272727
Processing trajectory for bias = 0.7954545454545454
Processing trajectory for bias = 0.8181818181818181
Processing trajectory for bias = 0.8409090909090909
Processing trajectory for bias = 0.8636363636363636
Processing trajectory for bias = 0.8863

Plot with shaded region for scaling graph

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# Base folder
base = "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_scaling/end_end_scaling/"

# Subfolders
folders = {
    "IM30": "Full_length",
    "IM30_H0_3": "H0_H3",
    "IM30_H4_6": "H4_H6"
}

# Bias values
bias_values = [
    "0_75","0_7727272727272727","0_7954545454545454","0_8181818181818181",
    "0_8409090909090909","0_8636363636363636","0_8863636363636364",
    "0_9090909090909091","0_9318181818181819","0_9545454545454546",
    "0_9772727272727273","1_0"
]

# Color scheme
dark_colors = {
    "IM30": "red",
    "IM30_H0_3": "green",
    "IM30_H4_6": "blue"
}

light_colors = {
    "IM30": "lightcoral",
    "IM30_H0_3": "palegreen",
    "IM30_H4_6": "lightskyblue"
}

def calculate_slope(x, y):
    if len(x) > 1:
        slope, intercept = np.polyfit(x, y, 1)
        return slope, intercept
    return None, None

results = {
    "bias": [],
    "slope_IM30": [],
    "slope_IM30_H0_3": [],
    "slope_IM30_H4_6": []
}

for bias in bias_values:

    plt.figure(figsize=(10,7))
    results["bias"].append(bias)

    for label, subfolder in folders.items():

        csv_file = f"R_vs_s_full_chain_bias_{bias}.csv"
        fpath = os.path.join(base, subfolder, csv_file)

        if not os.path.exists(fpath):
            print(f"⚠ File not found: {fpath}, skipping...")
            results[f"slope_{label}"].append(None)
            continue

        df = pd.read_csv(fpath)

        x_full = df["log(s)"]
        y_full = df["log(R_g(s)_mean)"]
        yerr_full = df["log(R_g(s)_std_rel)"] / np.sqrt(12)

        # Fit region mask
        mask = (df["log(s)"] >= 1) & (df["log(s)"] <= 3)
        x_fit = x_full[mask]
        y_fit = y_full[mask]

        # Plot data with SEM error bars
        plt.errorbar(
            x_full, y_full, yerr=yerr_full,
            marker='o', markersize=4,
            linewidth=2.0, elinewidth=1.5, capsize=3,
            color=dark_colors[label],
            ecolor=light_colors[label],
            label=label.replace("_", " ")
        )

        # Fit slope
        slope, intercept = calculate_slope(x_fit, y_fit)
        results[f"slope_{label}"].append(slope)

        # Plot fitted line in fit region only
        if slope is not None:
            x_line = np.linspace(1, 3, 100)
            y_line = slope * x_line + intercept
            plt.plot(
                x_line, y_line,
                color=dark_colors[label],
                linewidth=2.5,
                linestyle='--'
            )

        print(f"Bias {bias} | {label} | slope = {slope}")

    # Title with formatted %
    bias_float = float(bias.replace("_", "."))
    plt.title(f"{bias_float*100:.0f}% H-bond Strength",
              fontsize=30, fontweight="bold")

    # Axis labels
    plt.xlabel("log(s)", fontsize=30, fontweight="bold")
    plt.ylabel("log(R(s))", fontsize=30, fontweight="bold")

    # Axis limits
    plt.ylim(0, 5)

    # -----------------------------
    # Highlight fitting region
    # -----------------------------
    plt.axvspan(1, 3, alpha=0.08, color='gray', zorder=0)
    plt.axvline(1, linestyle=':', linewidth=2, color='black')
    plt.axvline(3, linestyle=':', linewidth=2, color='black')
    # -----------------------------

    # Ticks
    plt.xticks(fontsize=26, fontweight='bold')
    plt.yticks(fontsize=26, fontweight='bold')

    # Bold legend
    legend = plt.legend(fontsize=18, frameon=False)
    for text in legend.get_texts():
        text.set_fontweight('bold')

    # Thick border box
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    plt.tight_layout()

    plt.savefig(f"plot_bias_{bias}_styled.png", dpi=600)
    plt.close()

# Save slope table
df_out = pd.DataFrame(results)
df_out.to_csv("all_slopes_1_to_3.csv", index=False)

print("\nSaved 12 styled plots with shaded fit region + CSV file: all_slopes_1_to_3.csv\n")


Plot for Slope of shaded region

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load slope CSV
df = pd.read_csv("all_slopes_1_to_3.csv")

# Convert bias string like "0_75" → float
df["bias_value"] = df["bias"].str.replace("_", ".").astype(float)

# Convert to %
df["bias_percent"] = df["bias_value"] * 100

# Sort properly
df = df.sort_values("bias_percent")

# Colors
dark_colors = {
    "IM30": "red",
    "IM30_H0_3": "green",
    "IM30_H4_6": "blue"
}

plt.figure(figsize=(10,7))

# Plot curves
plt.plot(
    df["bias_percent"],
    df["slope_IM30"],
    marker='o',
    linewidth=2.5,
    markersize=7,
    color=dark_colors["IM30"],
    label="IM30"
)

plt.plot(
    df["bias_percent"],
    df["slope_IM30_H0_3"],
    marker='o',
    linewidth=2.5,
    markersize=7,
    color=dark_colors["IM30_H0_3"],
    label="IM30 H0–3"
)

plt.plot(
    df["bias_percent"],
    df["slope_IM30_H4_6"],
    marker='o',
    linewidth=2.5,
    markersize=7,
    color=dark_colors["IM30_H4_6"],
    label="IM30 H4–6"
)

# Labels
plt.xlabel("H-bond Strength (%)", fontsize=30, fontweight="bold")
plt.ylabel("Scaling Exponent (ν)", fontsize=30, fontweight="bold")

# Title
#plt.title("Scaling Exponent vs H-bond Strength",fontsize=30, fontweight="bold")

# Ticks
plt.xticks(fontsize=26, fontweight='bold')
plt.yticks(fontsize=26, fontweight='bold')

# Bold legend
legend = plt.legend(fontsize=20, frameon=False)
for text in legend.get_texts():
    text.set_fontweight('bold')

# Thick box
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()

plt.savefig("slope_vs_bias_percent_styled.png", dpi=600)
plt.show()
